Importing Libraries and Loading the Dataset

Link da dataset: https://www.kaggle.com/datasets/fredericods/ptbr-sentiment-analysis-datasets?resource=download

Nome do arquivo: b2w.csv

In [21]:
import pandas as pd
import re

In [2]:
df = pd.read_csv(r'G:\Meu Drive\Estudos Dados\NLP\b2w.csv')

In [6]:
df.head()

,original_index,review_text,review_text_processed,review_text_tokenized,polarity,rating,kfold_polarity,kfold_rating
0,11955,Bem macio e felpudo...recomendo. Preço imbatí...,bem macio e felpudo...recomendo. preco imbati...,"['bem', 'macio', 'felpudo', 'recomendo', 'prec...",1.0,4,1,1
1,35478,Produto excepcional! recomendo!!! inovador e ...,produto excepcional! recomendo!!! inovador e ...,"['produto', 'excepcional', 'recomendo', 'inova...",1.0,5,1,1
2,122760,recebi o produto antes do prazo mas veio com d...,recebi o produto antes do prazo mas veio com d...,"['recebi', 'produto', 'antes', 'do', 'prazo', ...",0.0,1,1,1
3,17114,Bom custo beneficio. Adequado para pessoas que...,bom custo beneficio. adequado para pessoas que...,"['bom', 'custo', 'beneficio', 'adequado', 'par...",1.0,5,1,1
4,19112,Além de higiênico tem o tamanho ideal. Só falt...,alem de higienico tem o tamanho ideal. so falt...,"['alem', 'de', 'higienico', 'tem', 'tamanho', ...",NaN,3,-1,1


In [5]:
# Lets inspect the dataset
print("\nFormato do dataset:", df.shape)
print("\nColunas disponíveis:", df.columns.tolist())


Formato do dataset: (132373, 8)

Colunas disponíveis: ['original_index', 'review_text', 'review_text_processed', 'review_text_tokenized', 'polarity', 'rating', 'kfold_polarity', 'kfold_rating']


Dealing with missing values in 'polarity'

In [7]:
df.isna().sum()

original_index               0
review_text                  0
review_text_processed        0
review_text_tokenized        0
polarity                 16315
rating                       0
kfold_polarity               0
kfold_rating                 0
dtype: int64

In [9]:
df.shape[0]/df.isna().sum()['polarity']

8.11357646337726

In [10]:
df = df.dropna(subset=["polarity"])

In [11]:
# Verifying the distribution of the polarity
print("\nDistribuição da polarity:")
print(df["polarity"].value_counts())


Distribuição da polarity:
polarity
1.0    80300
0.0    35758
Name: count, dtype: int64


Creating a Label Column (positive x negative)

In [12]:
def map_polarity_to_label(value):
    if value == 1.0:
        return "Positivo"
    elif value == 0.0:
        return "Negativo"
    return "Neutro"  # If there was another category, it would be considered neutral

df["sentiment_label"] = df["polarity"].apply(map_polarity_to_label)

print(df[["polarity", "sentiment_label"]].head(10))

    polarity sentiment_label
0        1.0        Positivo
1        1.0        Positivo
2        0.0        Negativo
3        1.0        Positivo
5        1.0        Positivo
6        1.0        Positivo
7        1.0        Positivo
8        1.0        Positivo
9        1.0        Positivo
10       0.0        Negativo


C:\Users\Bruno\AppData\Local\Temp\ipykernel_10728\2579390866.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["sentiment_label"] = df["polarity"].apply(map_polarity_to_label)


Choosing the Text Column for Sorting

The dataset offers:

review_text (raw text)
review_text_processed (text with some previous processing)
You can choose to work directly with review_text_processed, which can already come without punctuation or stopwords. If you want to apply your own pre-processing, use review_text and customize the steps. Below, we'll assume that we'll work with review_text_processed for illustrative purposes, but we'll show you how to add an additional cleaning step if desired.

In [ ]:
# If the 'review_text_processed' column is clean enough, we can use it:
df["text"] = df["review_text_processed"].astype(str)

# If you want to apply extra cleanup, you can create a function and apply it to df[“text”]:
def extra_cleaning(text):
    # Example: removing duplicate ellipses, possible strange symbols, etc.
    text = re.sub(r"\.{2,}", ".", text)  # transform ... into .
    # any other rules you want
    return text.lower()

df["text"] = df["text"].apply(extra_cleaning)

# Viewing:
print(df[["review_text_processed", "text"]].sample(5))

C:\Users\Bruno\AppData\Local\Temp\ipykernel_10728\3172243983.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["text"] = df["review_text_processed"].astype(str)


                                   review_text_processed  \
14679  potente e pratico de montar e usar. correspond...   
28494  o suporte e super robusto e visualmente muito ...   
73536  o cortador de grama e muito bom, pratico de se...   
55061  atendeu todas minhas necessidades e um pouco m...   
12851  compra realizada com sucesso,  otimo atendimen...   

                                                    text  
14679  potente e pratico de montar e usar. correspond...  
28494  o suporte e super robusto e visualmente muito ...  
73536  o cortador de grama e muito bom, pratico de se...  
55061  atendeu todas minhas necessidades e um pouco m...  
12851  compra realizada com sucesso,  otimo atendimen...  


C:\Users\Bruno\AppData\Local\Temp\ipykernel_10728\3172243983.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["text"] = df["text"].apply(extra_cleaning)


Let's separate the columns we're interested in: X (text) and y (label). Next, we'll divide them into training (70%) and test (30%).

In [16]:
from sklearn.model_selection import train_test_split

X = df["text"]
y = df["sentiment_label"]

# Stratified division to maintain class proportions
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print("Tamanho do treino:", X_train.shape[0])
print("Tamanho do teste:", X_test.shape[0])

Tamanho do treino: 81240
Tamanho do teste: 34818


Building the Pipeline (TF-IDF + Classifier)

Installation of Stopwords in Portuguese (NLTK)

In [17]:
import nltk
nltk.download("stopwords")
nltk.download("rslp")  # For stemmer in Portuguese, if you wish

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Bruno\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package rslp to
[nltk_data]     C:\Users\Bruno\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping stemmers\rslp.zip.


True

TF-IDF Pipeline + Logistic Regression

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
from nltk.corpus import stopwords

stopwords_pt = set(stopwords.words("portuguese"))

# Example of a simple cleaning function (if you want to remove stopwords in the pipeline)
def tokenizer(text):
    tokens = text.split()
    # Remove stopwords
    tokens = [t for t in tokens if t not in stopwords_pt]
    return tokens

# Pipeline Creation
pipeline = Pipeline([
    (
        "tfidf", 
        TfidfVectorizer(
            tokenizer=tokenizer,  # we can provide the tokenizer function
            ngram_range=(1,2),    # Considering unigrams and bigrams (e.g. “very good”)
            max_features=20000    # Adjust according to memory availability
        )
    ),
    ("clf", LogisticRegression(max_iter=300))
])

# Training
pipeline.fit(X_train, y_train)

# Predicting
y_pred = pipeline.predict(X_test)

c:\Users\Bruno\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\feature_extraction\text.py:525: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Acurácia do modelo: 0.9419
Relatório de Classificação:
              precision    recall  f1-score   support

    Negativo       0.91      0.90      0.91     10728
    Positivo       0.96      0.96      0.96     24090

    accuracy                           0.94     34818
   macro avg       0.93      0.93      0.93     34818
weighted avg       0.94      0.94      0.94     34818



Results

In [20]:
# Metrics
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.4f}")

print("Classification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.9419
Classification Report:
              precision    recall  f1-score   support

    Negativo       0.91      0.90      0.91     10728
    Positivo       0.96      0.96      0.96     24090

    accuracy                           0.94     34818
   macro avg       0.93      0.93      0.93     34818
weighted avg       0.94      0.94      0.94     34818



Accuracy: We get the proportion of correct predictions out of the total. Above 85%-90% is usually a good initial result, depending on how balanced the data set is and the complexity of the text.

Classification Report:
* Precision: of the items predicted as positive (or negative), how many actually belong to that class?
* Recall: of all the positive (or negative) items, how many were correctly detected?
* F1-score: harmonic mean between precision and recall.

If the dataset is unbalanced (many more examples of positive than negative), analyzing these metrics is essential to make sure that the model isn't just tending towards the majority class.

Optimization Tips

* Grid Search or Random Search: Adjust hyperparameters of the TfidfVectorizer (e.g. ngram_range, max_df, min_df) and the classifier (e.g. C for LogReg).
* Cross-validation: Use k-fold cross-validation to obtain a more robust estimate of performance.
* Advanced Models: Try transform-based architectures, such as BERTimbau or CamemBERT for Portuguese, which generally offer better performance than bag-of-words approaches.

Test with Cross Validation

In [19]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(pipeline, X, y, cv=5, scoring="accuracy")
print("Accuracy em cada fold:", scores)
print("Média de accuracy:", scores.mean())

c:\Users\Bruno\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\feature_extraction\text.py:525: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\Bruno\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\feature_extraction\text.py:525: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\Bruno\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\feature_extraction\text.py:525: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\Bruno\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\feature_extraction\text.py:525: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\Bruno\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\feature_extraction\text.py:525: Use

Accuracy em cada fold: [0.94308978 0.943736   0.94227124 0.94343199 0.94377666]
Média de accuracy: 0.9432611339708001


Conclusion
* Using real datasets: We demonstrated how to use Kaggle's ptbr-sentiment-analysis-datasets to build a sentiment classifier in Portuguese.
* The results both with a single dataset and using different folds showed that the model was able to recognize the text patterns in a way that performed very well, as well as being able to predict both classes with good accuracy, even with a difference in proportion between the predicted labels.

NLP process:
* Data cleaning and preparation (removal of rows with NaN, label mapping).
* Construction of a pipeline with TfidfVectorizer and a simple classifier (Logistic Regression).
* Model evaluation via accuracy, precision, recall and f1-score.
* Extensions: To improve, consider adjusting hyperparameters, pre-trained embeddings (BERT, GPT etc.), more refined removal of stopwords, lemmatization techniques specific to Portuguese, among others.